# 03 · Results
Builds the comparison table and plots from the files on Drive. This notebook needs no GPU.

Claims tested (from arXiv:2506.03230):
* **C1:** DiaBlo matches or beats LoRA at a comparable trainable-parameter budget. In the paper, Llama2-7B trained on MetaMathQA scores 66.5 on GSM8k with DiaBlo N=64 (1.04% params), against 60.6 for LoRA r=64 (1.67%).
* **C2:** Accuracy improves as N decreases (more parameters) but "plateaus around N=64" (App. A.3.5).
* **C3:** DiaBlo "achieves strong accuracy across a wide range of learning rates" (App. A.3.5, Fig. 5, on GSM8k).

Scope: GSM8k-train → GSM8k-test on Qwen2.5-1.5B with a single seed unless R5 was run. This is a **method-level reproduction at 1.5B scale**, not a reproduction of the paper's absolute numbers.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, os, glob
import pandas as pd
DRIVE = "/content/drive/MyDrive/diablo"; RESULTS = f"{DRIVE}/results"; EVALS = f"{DRIVE}/eval"
BACKEND = "vllm"   # whichever backend was used for the full eval

rows = []
for f in sorted(glob.glob(f"{EVALS}/R*_{BACKEND}_summary.json")):
    rid = os.path.basename(f).split("_")[0]
    if "parity" in f: continue
    s = json.load(open(f))
    row = dict(run=rid, acc=100*s["accuracy"], acc_orig=100*s["accuracy_orig_protocol"], n=s["n"])
    cfgf = f"{RESULTS}/{rid}/run_config.json"
    if os.path.exists(cfgf):
        c = json.load(open(cfgf))
        row.update(method=c["method"], size=(f"N={c['num_blocks']}" if c["method"]=="diablo" else f"r={c['lora_r']}"),
                   lr=c["lr"], seed=c["seed"], params_M=c["trainable_params"]/1e6)
        lh = json.load(open(f"{RESULTS}/{rid}/log_history.json"))
        row["train_h"] = lh[-1].get("train_runtime", float("nan"))/3600
        row["final_loss"] = [h["loss"] for h in lh if "loss" in h][-1] if any("loss" in h for h in lh) else None
    else:
        row.update(method="base", size="-", params_M=0.0)
    rows.append(row)
df = pd.DataFrame(rows).sort_values("run"); df

In [ ]:
# Loss curves (from the last checkpoint's trainer_state, which has the per-10-step losses)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
for rid in df.run:
    cks = sorted(glob.glob(f"{RESULTS}/{rid}/checkpoint-*/trainer_state.json"), key=lambda p: int(p.split("-")[-1].split("/")[0]))
    if not cks: continue
    h = [x for x in json.load(open(cks[-1]))["log_history"] if "loss" in x]
    ax.plot([x["step"] for x in h], [x["loss"] for x in h], label=rid)
ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.legend(); ax.set_title("Training loss"); plt.show()

In [ ]:
# Accuracy vs trainable parameters (C1, C2)
fig, ax = plt.subplots(figsize=(6,4))
for m, mk in [("diablo","o"),("lora","s")]:
    d = df[df.method==m].sort_values("params_M")
    ax.plot(d.params_M, d.acc, mk+"-", label=m)
    for _, r in d.iterrows(): ax.annotate(f"{r['run']} {r['size']} lr={r['lr']}", (r.params_M, r.acc), fontsize=7)
base = df[df.method=="base"]
if len(base): ax.axhline(base.acc.iloc[0], ls="--", c="gray", label="base (no FT)")
ax.set_xlabel("trainable params (M)"); ax.set_ylabel("GSM8k acc (%)"); ax.legend(); plt.show()

In [ ]:
# Paired significance (McNemar exact test) for DiaBlo vs LoRA on the same 1319 questions
from scipy.stats import binomtest
def load(rid): return {r["idx"]: r["correct"] for r in map(json.loads, open(f"{EVALS}/{rid}_{BACKEND}.jsonl"))}
def mcnemar(a, b):
    A, B = load(a), load(b)
    only_a = sum(A[i] and not B[i] for i in A); only_b = sum(B[i] and not A[i] for i in A)
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    print(f"{a} vs {b}: {a}-only correct {only_a}, {b}-only correct {only_b}, p = {p:.4f}")
for a, b in [("R1","R2"), ("R3","R8"), ("R1","R6"), ("R1","R3"), ("R1","R4")]:
    if os.path.exists(f"{EVALS}/{a}_{BACKEND}.jsonl") and os.path.exists(f"{EVALS}/{b}_{BACKEND}.jsonl"):
        mcnemar(a, b)

## Write-up template
| Claim | Paper evidence | Our result (Qwen2.5-1.5B, GSM8k) | Verdict |
|---|---|---|---|
| C1 DiaBlo ≥ LoRA at matched params | Table 2 (MetaMathQA, Llama2-7B) | R1 vs R2 (and R3 vs R8): Δacc, McNemar p | reproduced / not / inconclusive |
| C2 Gains plateau around N=64 | App. A.3.5, Table 10 | R3 / R1 / R4 | |
| C3 Robust to LR | App. A.3.5, Fig. 5 | R1 vs R7 | |

Deviations from the paper to report: a smaller model (Qwen2.5-1.5B rather than Llama), GSM8k training rather than MetaMathQA, fp16 on a T4 (the paper uses bf16), a single seed, and LoRA at a matched parameter count rather than r=64.